In [ ]:
import pandas as pd
import re

In [ ]:
# 1. Đọc file danh_sach_ma_distinct.txt để tạo bản đồ ánh xạ (Mapping)
groups_map = {}
with open('danh_sach_ma_distinct.txt', 'r', encoding='utf-8') as f:
    content = f.read()
    # Tìm các cụm mã nằm trong dấu ngoặc đơn () kể cả trên nhiều dòng
    matches = re.findall(r'\((.*?)\)', content, re.DOTALL)
    for group_str in matches:
        # Tách các mã SKU bằng dấu phẩy và làm sạch khoảng trắng/xuống dòng
        skus = [sku.strip() for sku in group_str.replace('\n', ' ').split(',') if sku.strip()]
        if skus:
            # SỬA LỖI QUAN TRỌNG: Lấy skus để representative_name là kiểu CHUỖI (String)
            # Không được gán representative_name = skus (vì skus là một danh sách)
            representative_name = str(skus)
            for sku in skus:
                groups_map[sku] = representative_name

In [ ]:
# 2. Đọc dữ liệu từ file
df = pd.read_excel("Dữ liệu bán hàng từ 2023 đến 09.2025 -new.xlsx")

In [ ]:
# 3. Tạo cột 'group_name' và đảm bảo dữ liệu là chuỗi văn bản sạch
# Sử dụng .map() để thay thế SKU bằng mã đại diện đầu tiên.
# .fillna() giúp giữ nguyên mã SKU gốc nếu mã đó không nằm trong nhóm nào trong file .txt
df['group_name'] = df['fsku'].str.strip().map(groups_map).fillna(df['fsku'].str.strip()).astype(str)


In [ ]:
# 4. Gom nhóm và tính tổng sản lượng (qty) theo tháng (year_month)
# Lúc này group_name chắc chắn là String, lệnh groupby sẽ chạy thành công
final_result = df.groupby(['group_name', 'year_month'])['qty'].sum().reset_index()

In [ ]:
# 5. Xuất file kết quả với 3 cột chuẩn: group_name, year_month, qty
final_result.to_csv('Ket_Qua_Gom_Nhom_Thanh_Cong.csv', index=False, encoding='utf-8-sig')

print("Hoàn tất! Hệ thống đã gom nhóm thành công với tên nhóm là mã SKU đầu tiên.")

Hoàn tất! Hệ thống đã gom nhóm thành công với tên nhóm là mã SKU đầu tiên.
